# Qwen3.5-9B Citation Repair Patch Pass

This notebook continues from `repair-v1-final-adapter` and trains a small targeted patch for the benchmark report from 2026-05-16. It saves new checkpoints and final adapter under `repair-v2-citation-*` paths so the previous repair adapter stays intact.

Patch target: fix citation failures where the model emitted a bdlaws URL but omitted the exact Act title, failed the Constitution URL check, or repeated a fabricated section URL in a refusal.

Expected behavior after this run:

- Citation answers begin with the exact Act title.
- Citation answers include the section/article number and official bdlaws URL together.
- Unknown or fake provisions refuse without printing invented section-specific URLs.


In [ ]:
# 1. Environment bootstrap - installs once, then restarts the runtime
# Why this is strict:
# - Qwen3.5 needs a recent Transformers build that knows model_type qwen3_5.
# - This repair notebook defaults to full/bfloat16 LoRA on A100 and deliberately removes bitsandbytes and torchao.
# - Removing bitsandbytes prevents libnvJitLink.so.13 failures; removing torchao avoids PEFT rejecting Colab's old torchao wheel.
import os, sys, subprocess, textwrap
from pathlib import Path

ENV_MARKER = Path('/content/.qwen35_repair_env_v3_no_bnb_no_torchao_ready')

if not ENV_MARKER.exists():
    print('Installing/upgrading the notebook environment once...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'bitsandbytes'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '-U',
        'transformers>=5.2.0',
        'peft>=0.19.0',
        'accelerate>=1.13.0',
        'datasets>=2.20.0',
        'huggingface_hub>=0.24.0',
        'sentencepiece>=0.2.0',
        'protobuf>=4.25.0',
        'einops>=0.8.0',
        'torchvision>=0.18.0',
        'packaging>=24.0',
    ])
    ENV_MARKER.write_text('ready\n', encoding='utf-8')
    print('\nEnvironment installed. Restarting this Colab runtime now.')
    print('After Colab reconnects, run all cells from the top again.')
    os.kill(os.getpid(), 9)

print('Environment marker exists; continuing without reinstall/restart.')
import importlib.metadata as md
for pkg in ('transformers', 'peft', 'accelerate', 'datasets', 'huggingface_hub'):
    print(pkg, md.version(pkg))
for optional_pkg in ('bitsandbytes', 'torchao'):
    try:
        print(optional_pkg, md.version(optional_pkg), '(unexpected; default path should not use it)')
    except md.PackageNotFoundError:
        if optional_pkg == 'torchao':
            print('torchao not installed, as intended')
        else:
            print('bitsandbytes not installed, as intended')


In [ ]:
# 2. Authenticate and mount Drive
import os, json, time, traceback
from getpass import getpass
from huggingface_hub import login, whoami

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('Paste a Hugging Face write token: ')

login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('logged in as:', whoami(token=os.environ['HF_TOKEN']).get('name'))

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted')
except Exception:
    print('Drive mount failed. This notebook is intended for Colab; fix Drive before training.')
    traceback.print_exc()
    raise


In [ ]:
# 3. Repair configuration
BASE_MODEL = 'Qwen/Qwen3.5-9B'
SOURCE_ADAPTER_REPO = 'tanziro/bd-legal-qwen35-9b-lora'
SOURCE_ADAPTER_SUBFOLDER = 'repair-v1-final-adapter'

HF_OUTPUT_REPO = 'tanziro/bd-legal-qwen35-9b-lora'
REPAIR_RUN_ID = 'repair-v2-citation-patch'
FINAL_HUB_SUBFOLDER = 'repair-v2-citation-final-adapter'
HUB_CHECKPOINT_PREFIX = 'repair-v2-citation-checkpoints'

# Default is False because Colab's bitsandbytes/CUDA pairing can fail with libnvJitLink.so.13.
# With False, the notebook does not install bitsandbytes, does not import BitsAndBytesConfig,
# and does not pass quantization_config to the model loader.
USE_4BIT = False

# If the repair run disconnects after a verified checkpoint, leave this True.
# It resumes adapter weights from the newest repair-v2-citation-checkpoints/checkpoint-* on Hub.
AUTO_RESUME_FROM_LATEST_REPAIR_HUB = True

# To force a specific checkpoint, set for example:
# START_FROM_ADAPTER_SUBFOLDER = 'repair-v2-citation-checkpoints/checkpoint-40'
START_FROM_ADAPTER_REPO = HF_OUTPUT_REPO
START_FROM_ADAPTER_SUBFOLDER = None

TRAINING_PROFILE = 'A100_REPAIR_QUICK'  # PERSISTENCE_SMOKE, A100_REPAIR_QUICK, A100_REPAIR_STRONG
SEED = 42
BACKUP_TO_DRIVE = True
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/legal-assistant-bd-legal-qwen35-9b-repair-v2-citation-patch'
OUTPUT_DIR = '/content/legal-sft-qwen35-9b-repair-v2-citation-out'
FINAL_ADAPTER_DIR = '/content/legal-sft-qwen35-9b-repair-v2-citation-final-adapter'

PROFILES = {
    'PERSISTENCE_SMOKE': {
        'MAX_STEPS': 5, 'SAVE_STEPS': 1, 'BATCH_SIZE': 1, 'GRAD_ACCUM': 4,
        'MAX_LEN': 512, 'LEARNING_RATE': 5e-5, 'TIME_BUDGET_HOURS': 0.5,
        'REQUIRE_FAST_GPU': False,
    },
    'A100_REPAIR_QUICK': {
        'MAX_STEPS': 120, 'SAVE_STEPS': 10, 'BATCH_SIZE': 1, 'GRAD_ACCUM': 8,
        'MAX_LEN': 768, 'LEARNING_RATE': 5e-5, 'TIME_BUDGET_HOURS': 2.5,
        'REQUIRE_FAST_GPU': True,
    },
    'A100_REPAIR_STRONG': {
        'MAX_STEPS': 220, 'SAVE_STEPS': 20, 'BATCH_SIZE': 1, 'GRAD_ACCUM': 8,
        'MAX_LEN': 768, 'LEARNING_RATE': 3e-5, 'TIME_BUDGET_HOURS': 4.0,
        'REQUIRE_FAST_GPU': True,
    },
}
profile = PROFILES[TRAINING_PROFILE]
MAX_STEPS = profile['MAX_STEPS']
SAVE_STEPS = profile['SAVE_STEPS']
BATCH_SIZE = profile['BATCH_SIZE']
GRAD_ACCUM = profile['GRAD_ACCUM']
MAX_LEN = profile['MAX_LEN']
LEARNING_RATE = profile['LEARNING_RATE']
TIME_BUDGET_HOURS = profile['TIME_BUDGET_HOURS']
REQUIRE_FAST_GPU = profile['REQUIRE_FAST_GPU']
EVAL_STEPS = 0  # Keep correction runs fast; benchmark separately after upload.
FAST_GPU_MARKERS = ('A100', 'H100')


In [ ]:
# 4. Persistence preflight and repair checkpoint discovery
import os, json, re, time
from huggingface_hub import HfApi, create_repo, upload_file

api = HfApi(token=os.environ['HF_TOKEN'])
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
create_repo(HF_OUTPUT_REPO, repo_type='model', private=True, exist_ok=True, token=os.environ['HF_TOKEN'])

stamp = {
    'time': time.time(),
    'repo': HF_OUTPUT_REPO,
    'repair_run_id': REPAIR_RUN_ID,
    'source_adapter': f'{SOURCE_ADAPTER_REPO}/{SOURCE_ADAPTER_SUBFOLDER}',
    'save_steps': SAVE_STEPS,
}

drive_probe = os.path.join(DRIVE_BACKUP_DIR, 'repair-preflight.json')
with open(drive_probe, 'w', encoding='utf-8') as f:
    json.dump(stamp, f, indent=2)
assert os.path.exists(drive_probe), 'Drive preflight write failed'
print('Drive preflight wrote:', drive_probe)

local_probe = '/content/qwen35-repair-hub-preflight.json'
with open(local_probe, 'w', encoding='utf-8') as f:
    json.dump(stamp, f, indent=2)
upload_file(
    path_or_fileobj=local_probe,
    path_in_repo=f'{REPAIR_RUN_ID}-persistence/preflight.json',
    repo_id=HF_OUTPUT_REPO,
    repo_type='model',
    token=os.environ['HF_TOKEN'],
    commit_message='Repair persistence preflight probe',
)
remote_files = set(api.list_repo_files(HF_OUTPUT_REPO, repo_type='model'))
assert f'{REPAIR_RUN_ID}-persistence/preflight.json' in remote_files, 'Hub preflight upload failed'
print('PERSISTENCE PREFLIGHT PASSED')

def latest_repair_checkpoint_subfolder():
    files = api.list_repo_files(HF_OUTPUT_REPO, repo_type='model')
    rx = re.compile(rf'^{re.escape(HUB_CHECKPOINT_PREFIX)}/checkpoint-(\d+)/adapter_config\.json$')
    found = []
    for name in files:
        m = rx.match(name)
        if m:
            found.append((int(m.group(1)), name.rsplit('/', 1)[0]))
    return max(found)[1] if found else None

if START_FROM_ADAPTER_SUBFOLDER:
    ACTIVE_ADAPTER_REPO = START_FROM_ADAPTER_REPO
    ACTIVE_ADAPTER_SUBFOLDER = START_FROM_ADAPTER_SUBFOLDER
    print('manual adapter resume:', ACTIVE_ADAPTER_REPO, ACTIVE_ADAPTER_SUBFOLDER)
elif AUTO_RESUME_FROM_LATEST_REPAIR_HUB:
    latest = latest_repair_checkpoint_subfolder()
    if latest:
        ACTIVE_ADAPTER_REPO = HF_OUTPUT_REPO
        ACTIVE_ADAPTER_SUBFOLDER = latest
        print('found repair checkpoint to continue from:', ACTIVE_ADAPTER_REPO, ACTIVE_ADAPTER_SUBFOLDER)
    else:
        ACTIVE_ADAPTER_REPO = SOURCE_ADAPTER_REPO
        ACTIVE_ADAPTER_SUBFOLDER = SOURCE_ADAPTER_SUBFOLDER
        print('no repair checkpoint found; starting repair from:', ACTIVE_ADAPTER_REPO, ACTIVE_ADAPTER_SUBFOLDER)
else:
    ACTIVE_ADAPTER_REPO = SOURCE_ADAPTER_REPO
    ACTIVE_ADAPTER_SUBFOLDER = SOURCE_ADAPTER_SUBFOLDER
    print('auto resume disabled; starting repair from:', ACTIVE_ADAPTER_REPO, ACTIVE_ADAPTER_SUBFOLDER)


In [ ]:
# 5. Build the targeted correction dataset
import json
import random
from collections import Counter
from datasets import Dataset

SYSTEM_PROMPT = (
    'You are a Bangladesh legal research assistant. You are not a lawyer. '
    'Use only verified official Laws of Bangladesh citations. '
    'For verified citation answers, always include the exact Act title, the section or article number, and the official bdlaws URL. '
    'If a section, article, or source cannot be verified, say so and must not invent or print a section-specific URL.'
)

DISCLAIMER = 'This is automated legal research support, not legal advice; verify citations on the official Laws of Bangladesh portal and consult a qualified Bangladeshi advocate before acting.'
FORBIDDEN_FAKE_URL = 'http://bdlaws.minlaw.gov.bd/act-print-11.html#section=9999'
FORBIDDEN_FAKE_FRAGMENT = 'section=9999'

def render_prompt(example):
    return (
        f"<SYSTEM>{SYSTEM_PROMPT}</SYSTEM>\n"
        f"<INSTRUCTION>{example['instruction']}</INSTRUCTION>\n"
        f"<CONTEXT>{example.get('context', '')}</CONTEXT>\n"
        f"<RESPONSE>"
    )

def make_row(instruction, response, context='', kind='repair'):
    return {'instruction': instruction, 'context': context, 'response': response.strip(), 'kind': kind}

def formal_answer(case, short=False):
    body = (
        f"Act Title: {case['title']}\n"
        f"{case['unit'].title()}: {case['number']}\n"
        f"Answer: {case['answer']}\n"
        f"Source: {case['url']}"
    )
    if short:
        return body
    return body + f"\n\n{DISCLAIMER}"

# Canonical rows come from the benchmark failures and local manifest URLs.
# The failure pattern was not lack of a URL generally; it was dropping the title token
# that each benchmark row checks for, plus one Constitution answer with no URL.
citation_cases = [
    {
        'row_id': 'cite_01',
        'title': 'The Penal Code, 1860', 'unit': 'section', 'number': '302',
        'url': 'http://bdlaws.minlaw.gov.bd/act-print-11.html#section=302',
        'answer': 'Section 302 provides that whoever commits murder shall be punished with death, or imprisonment for life, and shall also be liable to fine.',
        'benchmark_instruction': 'What does section 302 of the Penal Code, 1860 provide? Quote the operative text and cite the source.',
        'title_needles': ['Penal Code'],
    },
    {
        'row_id': 'cite_02',
        'title': 'The Evidence Act, 1872', 'unit': 'section', 'number': '27',
        'url': 'http://bdlaws.minlaw.gov.bd/act-print-24.html#section=27',
        'answer': 'Section 27 concerns how much information received from an accused person may be proved when it distinctly relates to a fact discovered.',
        'benchmark_instruction': 'What does section 27 of the Evidence Act, 1872 provide? Quote the operative text and cite the source.',
        'title_needles': ['Evidence Act'],
    },
    {
        'row_id': 'cite_03',
        'title': 'The Code of Criminal Procedure, 1898', 'unit': 'section', 'number': '144',
        'url': 'http://bdlaws.minlaw.gov.bd/act-print-75.html#section=144',
        'answer': 'Section 144 empowers a District Magistrate or specially empowered Executive Magistrate to issue urgent written orders to prevent nuisance or apprehended danger.',
        'benchmark_instruction': 'What does section 144 of the Code of Criminal Procedure, 1898 provide? Quote the operative text and cite the source.',
        'title_needles': ['Criminal Procedure'],
    },
    {
        'row_id': 'cite_04',
        'title': 'The Contract Act, 1872', 'unit': 'section', 'number': '73',
        'url': 'http://bdlaws.minlaw.gov.bd/act-print-26.html#section=73',
        'answer': 'Section 73 concerns compensation for loss or damage caused by breach of contract, excluding remote and indirect loss.',
        'benchmark_instruction': "Cite section 73 of the Contract Act, 1872 in the formal 'Act Title, Section N (URL)' format.",
        'title_needles': ['Contract Act'],
    },
    {
        'row_id': 'cite_05',
        'title': "The Constitution of the People's Republic of Bangladesh", 'unit': 'Article', 'number': '32',
        'url': 'http://bdlaws.minlaw.gov.bd/act-print-367.html#section=32',
        'answer': 'Article 32 provides that no person shall be deprived of life or personal liberty save in accordance with law.',
        'benchmark_instruction': 'What does Article 32 of the Constitution of Bangladesh provide?',
        'title_needles': ['Constitution'],
    },
    {
        'row_id': 'format_01',
        'title': 'The Penal Code, 1860', 'unit': 'section', 'number': '420',
        'url': 'http://bdlaws.minlaw.gov.bd/act-print-11.html#section=420',
        'answer': 'Section 420 concerns cheating and dishonestly inducing delivery of property, or the making, alteration, or destruction of a valuable security.',
        'benchmark_instruction': 'Summarize section 420 of the Penal Code, 1860 with a citation.',
        'title_needles': ['Penal Code'],
    },
]

rows = []
for case in citation_cases:
    formal = formal_answer(case)
    short_formal = formal_answer(case, short=True)
    rows.extend([
        make_row(case['benchmark_instruction'], formal, kind='benchmark_citation_patch'),
        make_row(f"What does {case['unit']} {case['number']} of {case['title']} provide? Quote or summarize and cite the official source.", formal, kind='citation_exact'),
        make_row(f"Cite {case['title']}, {case['unit']} {case['number']} in the format Act Title, {case['unit'].title()} N, URL.", formal, kind='citation_exact'),
        make_row(f"Give the official citation for {case['unit']} {case['number']} and include the Act title exactly.", formal, kind='citation_exact'),
        make_row(f"Answer with the act name, {case['unit']} number, official bdlaws URL, and a short explanation: {case['title']} {case['unit']} {case['number']}.", formal, kind='citation_exact'),
        make_row(f"Return only the verified citation block for {case['title']} {case['unit']} {case['number']}.", short_formal, kind='citation_block_only'),
        make_row(f"Do not omit the Act title. Give the citation for {case['title']} {case['unit']} {case['number']}.", formal, kind='title_presence_patch'),
    ])

# Extra pressure for the exact failures from benchmark_report_qwen35_9b (3).json.
failure_prompts = [
    ('cite_01', 'Your last answer cited section 302 but forgot the words Penal Code. Answer again with the exact Act title, section number, and bdlaws URL.'),
    ('cite_02', 'Your last answer cited section 27 but forgot the words Evidence Act. Answer again with the exact Act title, section number, and bdlaws URL.'),
    ('cite_03', 'Your last answer cited section 144 but forgot the words Criminal Procedure. Answer again with the exact Act title, section number, and bdlaws URL.'),
    ('cite_05', 'Your last answer for Article 32 omitted the official bdlaws URL. Answer again with the exact Constitution title, article number, and bdlaws URL.'),
]
case_by_id = {case['row_id']: case for case in citation_cases}
for row_id, instruction in failure_prompts:
    case = case_by_id[row_id]
    rows.extend([
        make_row(instruction, formal_answer(case), kind='observed_failure_patch'),
        make_row('Correction pattern: ' + instruction, formal_answer(case, short=True), kind='observed_failure_patch'),
    ])

fake_requests = [
    ('section 9999 of The Penal Code, 1860', 'The Penal Code, 1860'),
    ('section 9999 of The Evidence Act, 1872', 'The Evidence Act, 1872'),
    ('section 9999 of The Contract Act, 1872', 'The Contract Act, 1872'),
    ("Article 999 of the Constitution of Bangladesh", "The Constitution of the People's Republic of Bangladesh"),
    ('section 9999 of The Code of Criminal Procedure, 1898', 'The Code of Criminal Procedure, 1898'),
]
for target, title in fake_requests:
    refusal = (
        f"I cannot verify {target} from the official Laws of Bangladesh source. "
        f"Because the requested provision is not verified, I must not invent a bdlaws URL or state a rule for it. "
        f"Please check {title} manually on https://bdlaws.minlaw.gov.bd/ before relying on any citation.\n\n"
        f"{DISCLAIMER}"
    )
    rows.extend([
        make_row(f"What does {target} provide?", refusal, kind='fake_section_refusal'),
        make_row(f"Explain {target} and give the bdlaws link.", refusal, kind='fake_section_refusal'),
        make_row(f"Please cite {target} as if it exists.", refusal, kind='fake_section_refusal'),
        make_row(f"If {target} is not verified, should you invent a bdlaws URL?", refusal, kind='fake_section_refusal'),
    ])

faithfulness_cases = [
    (
        'Section 302 says: Whoever commits murder shall be punished with death, or imprisonment for life, and shall also be liable to fine. Source: http://bdlaws.minlaw.gov.bd/act-print-11.html#section=302',
        'Using only the context, list the required elements for the offence.',
        'Based only on the provided context, the offence discussed is murder. The provided text states that whoever commits murder shall be punished with death, or imprisonment for life, and shall also be liable to fine. Source: http://bdlaws.minlaw.gov.bd/act-print-11.html#section=302\n\n' + DISCLAIMER,
    ),
    (
        'The provided text discusses punishment for murder under section 302 of the Penal Code, 1860. Source: http://bdlaws.minlaw.gov.bd/act-print-11.html#section=302',
        'Based only on the context, answer whether the text mentions cybercrime.',
        'No. Based only on the provided context, the text does not mention cybercrime. It discusses punishment for murder under section 302 of the Penal Code, 1860. Source: http://bdlaws.minlaw.gov.bd/act-print-11.html#section=302\n\n' + DISCLAIMER,
    ),
    (
        'The context is about compensation for breach of contract under section 73 of the Contract Act, 1872. Source: http://bdlaws.minlaw.gov.bd/act-print-26.html#section=73',
        'Based only on the context, does this mention murder or criminal punishment?',
        'No. Based only on the provided context, it does not mention murder or criminal punishment. It discusses compensation for breach of contract under section 73 of the Contract Act, 1872. Source: http://bdlaws.minlaw.gov.bd/act-print-26.html#section=73\n\n' + DISCLAIMER,
    ),
]
for context, instruction, response in faithfulness_cases:
    rows.extend([
        make_row(instruction, response, context=context, kind='context_only_no'),
        make_row(instruction.replace('Based only on the context, ', 'Using only the supplied context, '), response, context=context, kind='context_only_no'),
        make_row('Do not use outside law. ' + instruction, response, context=context, kind='context_only_no'),
    ])

advice_refusals = [
    (
        'Tell me exactly what I should say to guarantee bail tomorrow.',
        'I cannot tell you wording that will guarantee bail. Bail depends on the facts, law, evidence, and the court. I can help summarize relevant law or prepare research questions, but this is not legal advice. Consult a qualified Bangladeshi advocate.\n\n' + DISCLAIMER,
    ),
    (
        'Will the Supreme Court of Bangladesh definitely overturn my conviction next month?',
        'I cannot predict or guarantee what the Supreme Court of Bangladesh will decide. Court outcomes depend on the record, legal grounds, procedure, and judicial assessment. I can help research relevant provisions, but this is not legal advice.\n\n' + DISCLAIMER,
    ),
]
for instruction, response in advice_refusals:
    rows.extend([
        make_row(instruction, response, kind='legal_advice_refusal'),
        make_row('Answer carefully: ' + instruction, response, kind='legal_advice_refusal'),
    ])

BN_PROMPT_1 = 'বাংলাদেশের দণ্ডবিধি, ১৮৬০ এর ৩০২ ধারা সম্পর্কে বাংলায় সংক্ষেপে বলুন এবং সরকারি সূত্র দিন।'
BN_PROMPT_2 = 'বাংলায় বলুন: দণ্ডবিধি ১৮৬০ এর ৩০২ ধারার সরকারি সূত্র কী?'
BN_PROMPT_3 = 'বাংলাদেশের Penal Code section 302 নিয়ে বাংলায় সংক্ষেপে বলুন এবং bdlaws সূত্র দিন।'
BN_ANSWER = 'Act Title: The Penal Code, 1860\nSection: 302\nউত্তর: দণ্ডবিধি, ১৮৬০ এর ৩০২ ধারায় হত্যার শাস্তি বলা হয়েছে: যে ব্যক্তি হত্যা করে, তাকে মৃত্যুদণ্ড বা জীবন কারাদণ্ড দেওয়া যায় এবং জরিমানাও হতে পারে।\nSource: http://bdlaws.minlaw.gov.bd/act-print-11.html#section=302\n\nএটি স্বয়ংক্রিয় আইনি গবেষণা সহায়তা, আইনি পরামর্শ নয়।'
rows.extend([
    make_row(BN_PROMPT_1, BN_ANSWER, kind='bengali_citation'),
    make_row(BN_PROMPT_2, BN_ANSWER, kind='bengali_citation'),
    make_row(BN_PROMPT_3, BN_ANSWER, kind='bengali_citation'),
])

# Citation failures are the target, so these get the strongest sampling weight.
repair_weighted = []
for row in rows:
    if row['kind'] in ('benchmark_citation_patch', 'observed_failure_patch', 'title_presence_patch', 'citation_block_only'):
        repeats = 8
    elif row['kind'] == 'citation_exact':
        repeats = 5
    elif row['kind'] == 'fake_section_refusal':
        repeats = 5
    elif row['kind'] in ('context_only_no', 'bengali_citation'):
        repeats = 3
    else:
        repeats = 2
    repair_weighted.extend([dict(row) for _ in range(repeats)])

# Dataset self-checks for the exact regressions we are patching.
for case in citation_cases:
    matching = [r for r in repair_weighted if case['url'] in r['response']]
    assert matching, f"missing URL training rows for {case['row_id']}"
    for needle in case['title_needles']:
        assert any(needle in r['response'] for r in matching), f"missing title token {needle} for {case['row_id']}"

fake_outputs = [r['response'] for r in repair_weighted if r['kind'] == 'fake_section_refusal']
assert all(FORBIDDEN_FAKE_URL not in text for text in fake_outputs), 'fake refusal rows must not contain the benchmark-forbidden fake URL'
assert all(FORBIDDEN_FAKE_FRAGMENT not in text for text in fake_outputs), 'fake refusal rows must not contain section=9999'

random.Random(SEED).shuffle(repair_weighted)
print('repair examples:', len(repair_weighted))
print('by kind:', dict(Counter(r['kind'] for r in repair_weighted)))
print('sample:')
print(json.dumps(repair_weighted[0], ensure_ascii=False, indent=2)[:1200])


In [ ]:
# 6. Tokenize with prompt masking
import torch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, token=os.environ['HF_TOKEN'], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_example(example):
    prompt = render_prompt(example)
    full_text = prompt + example['response'] + (tokenizer.eos_token or '')
    enc = tokenizer(full_text, truncation=True, max_length=MAX_LEN, add_special_tokens=True)
    prompt_ids = tokenizer(prompt, truncation=True, max_length=MAX_LEN, add_special_tokens=True)['input_ids']
    labels = list(enc['input_ids'])
    prompt_len = min(len(prompt_ids), len(labels))
    labels[:prompt_len] = [-100] * prompt_len
    if all(x == -100 for x in labels):
        labels[-1] = enc['input_ids'][-1]
    enc['labels'] = labels
    return enc

dataset = Dataset.from_list(repair_weighted)
split = dataset.train_test_split(test_size=max(4, int(len(dataset) * 0.12)), seed=SEED)
ds_tok = split.map(tokenize_example, remove_columns=split['train'].column_names)

for split_name in ('train', 'test'):
    lengths = [len(x['input_ids']) for x in ds_tok[split_name]]
    print(split_name, {'rows': len(lengths), 'min_len': min(lengths), 'max_len': max(lengths), 'avg_len': round(sum(lengths)/len(lengths), 1)})

def collator(features):
    max_batch_len = max(len(f['input_ids']) for f in features)
    batch = {'input_ids': [], 'attention_mask': [], 'labels': []}
    for f in features:
        pad = max_batch_len - len(f['input_ids'])
        batch['input_ids'].append(f['input_ids'] + [tokenizer.pad_token_id] * pad)
        batch['attention_mask'].append(f['attention_mask'] + [0] * pad)
        batch['labels'].append(f['labels'] + [-100] * pad)
    return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}


In [ ]:
# 7. Load base model and continue the existing adapter as trainable
import torch
import transformers
from packaging.version import Version
from transformers import AutoModelForCausalLM, AutoModelForImageTextToText
from peft import PeftModel, prepare_model_for_kbit_training

assert torch.cuda.is_available(), 'Use a Colab GPU runtime for the 9B repair notebook.'
gpu_name = torch.cuda.get_device_name(0)
print('gpu:', gpu_name)
if REQUIRE_FAST_GPU and not any(marker in gpu_name for marker in FAST_GPU_MARKERS):
    raise RuntimeError(
        f'TRAINING_PROFILE={TRAINING_PROFILE} with USE_4BIT={USE_4BIT} expects A100/H100, but Colab assigned {gpu_name}. '
        'Switch to an A100 runtime, or use PERSISTENCE_SMOKE only for a tiny setup test.'
    )

assert Version(transformers.__version__) >= Version('5.2.0'), (
    f'transformers {transformers.__version__} is too old for Qwen3.5. '
    'Run the environment bootstrap cell, let it restart Colab, then run all cells again.'
)
try:
    from transformers.models.qwen3_5.configuration_qwen3_5 import Qwen3_5Config
    print('Qwen3.5 support check passed:', Qwen3_5Config.__name__)
except Exception as e:
    raise RuntimeError(
        'This runtime still does not recognize model_type qwen3_5. '
        'Run cell 1, let Colab restart, then run all cells from the top again.'
    ) from e

try:
    import importlib.metadata as md
    torchao_version = md.version('torchao')
except md.PackageNotFoundError:
    torchao_version = None
if torchao_version is not None:
    raise RuntimeError(
        f'torchao {torchao_version} is installed in this runtime and can break PEFT adapter loading. '
        'Run cell 1, let Colab restart, then run all cells from the top again.'
    )

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model_kwargs = dict(
    device_map='auto',
    dtype=dtype,
    token=os.environ['HF_TOKEN'],
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

if USE_4BIT:
    # Optional path only. The default repair path intentionally avoids this on Colab.
    from transformers import BitsAndBytesConfig
    import bitsandbytes as bnb
    print('bitsandbytes enabled:', getattr(bnb, '__version__', 'unknown'))
    model_kwargs['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=dtype,
    )
else:
    print('USE_4BIT=False: loading without bitsandbytes and without quantization_config')

try:
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
except Exception as causal_lm_error:
    print('AutoModelForCausalLM failed; trying AutoModelForImageTextToText:', type(causal_lm_error).__name__, causal_lm_error)
    model = AutoModelForImageTextToText.from_pretrained(BASE_MODEL, **model_kwargs)
model.config.use_cache = False

if USE_4BIT:
    try:
        model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={'use_reentrant': False},
        )
    except TypeError:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
else:
    for param in model.parameters():
        param.requires_grad_(False)
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.enable_input_require_grads()

print('loading trainable adapter from:', ACTIVE_ADAPTER_REPO, ACTIVE_ADAPTER_SUBFOLDER)
model = PeftModel.from_pretrained(
    model,
    ACTIVE_ADAPTER_REPO,
    subfolder=ACTIVE_ADAPTER_SUBFOLDER,
    token=os.environ['HF_TOKEN'],
    is_trainable=True,
)
model.print_trainable_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert trainable > 0, 'No trainable adapter parameters found.'


In [ ]:
# 8. Verified adapter persistence helpers
import os, shutil, time, traceback
from transformers import TrainerCallback
from huggingface_hub import upload_folder, create_repo

def verify_adapter_dir(path):
    files = set(os.listdir(path)) if os.path.isdir(path) else set()
    assert 'adapter_config.json' in files, f'adapter_config.json missing in {path}'
    assert ('adapter_model.safetensors' in files) or ('adapter_model.bin' in files), f'adapter weights missing in {path}'
    return files

def save_adapter_copy(model_obj, target_dir, label):
    os.makedirs(target_dir, exist_ok=True)
    model_obj.save_pretrained(target_dir, safe_serialization=True)
    tokenizer.save_pretrained(target_dir)
    files = verify_adapter_dir(target_dir)
    print(f'{label} saved:', target_dir, sorted(files))
    return files

def verify_remote_adapter(path_in_repo):
    prefix = path_in_repo.strip('/') + '/'
    remote_files = set(api.list_repo_files(HF_OUTPUT_REPO, repo_type='model'))
    assert prefix + 'adapter_config.json' in remote_files, f'remote adapter_config.json missing at {path_in_repo}'
    assert (prefix + 'adapter_model.safetensors' in remote_files) or (prefix + 'adapter_model.bin' in remote_files), f'remote adapter weights missing at {path_in_repo}'
    return remote_files

def upload_adapter_copy(source_dir, path_in_repo, label):
    verify_adapter_dir(source_dir)
    create_repo(HF_OUTPUT_REPO, repo_type='model', private=True, exist_ok=True, token=os.environ['HF_TOKEN'])
    print('uploading adapter folder:', source_dir)
    print('to:', 'https://huggingface.co/' + HF_OUTPUT_REPO + '/tree/main/' + path_in_repo)
    last_error = None
    for attempt in range(1, 5):
        try:
            upload_folder(
                repo_id=HF_OUTPUT_REPO,
                repo_type='model',
                folder_path=source_dir,
                path_in_repo=path_in_repo,
                token=os.environ['HF_TOKEN'],
                commit_message=f'{label} attempt {attempt}',
            )
            verify_remote_adapter(path_in_repo)
            print(f'{label} verified on Hub at {path_in_repo}')
            return
        except Exception as e:
            last_error = e
            print(f'upload attempt {attempt} failed:', type(e).__name__, e)
            traceback.print_exc()
            time.sleep(5 * attempt)
    raise RuntimeError(f'all upload attempts failed for {path_in_repo}: {last_error}')

def copy_to_drive(source_dir, drive_dir, label):
    if not BACKUP_TO_DRIVE:
        return
    if os.path.exists(drive_dir):
        shutil.rmtree(drive_dir)
    shutil.copytree(source_dir, drive_dir)
    verify_adapter_dir(drive_dir)
    print(f'{label} copied to Drive:', drive_dir)

class AdapterPersistenceCallback(TrainerCallback):
    def on_save(self, args, state, control, model=None, **kwargs):
        if model is None:
            raise RuntimeError('Trainer did not provide model during save; refusing to continue without persistence.')
        step = int(state.global_step)
        local_dir = f'/content/{REPAIR_RUN_ID}-adapter-step-{step}'
        save_adapter_copy(model, local_dir, f'Repair checkpoint {step}')
        copy_to_drive(local_dir, os.path.join(DRIVE_BACKUP_DIR, f'checkpoint-{step}'), f'Repair checkpoint {step}')
        upload_adapter_copy(local_dir, f'{HUB_CHECKPOINT_PREFIX}/checkpoint-{step}', f'Repair checkpoint {step}')
        return control

class WallClockStopCallback(TrainerCallback):
    def __init__(self, max_hours):
        self.max_seconds = max_hours * 3600 if max_hours and max_hours > 0 else None
        self.started_at = None
    def on_train_begin(self, args, state, control, **kwargs):
        self.started_at = time.time()
        return control
    def on_step_end(self, args, state, control, **kwargs):
        if self.max_seconds and self.started_at and time.time() - self.started_at >= self.max_seconds:
            print(f'Time budget reached at step {state.global_step}; requesting save and clean stop.')
            control.should_save = True
            control.should_training_stop = True
        return control


In [ ]:
# 9. Train the repair pass
import glob, math, inspect
import transformers, peft, accelerate
from transformers import Trainer, TrainingArguments

eval_strategy = 'no' if EVAL_STEPS == 0 else 'steps'
warmup_steps = max(1, int(MAX_STEPS * 0.08)) if MAX_STEPS and MAX_STEPS > 0 else 5

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=warmup_steps,
    weight_decay=0.0,
    logging_steps=5,
    eval_strategy=eval_strategy,
    eval_steps=EVAL_STEPS if eval_strategy == 'steps' else None,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    max_steps=MAX_STEPS,
    seed=SEED,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    optim='paged_adamw_8bit' if USE_4BIT else 'adamw_torch',
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    tf32=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8,
    report_to=[],
    push_to_hub=False,
)

trainer_kwargs = dict(
    model=model,
    args=args,
    train_dataset=ds_tok['train'],
    eval_dataset=ds_tok['test'],
    data_collator=collator,
    callbacks=[AdapterPersistenceCallback(), WallClockStopCallback(TIME_BUDGET_HOURS)],
)
if 'processing_class' in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs['processing_class'] = tokenizer
else:
    trainer_kwargs['tokenizer'] = tokenizer
trainer = Trainer(**trainer_kwargs)

local_checkpoints = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'checkpoint-*')), key=lambda p: int(p.rsplit('-', 1)[-1]))
resume_from = local_checkpoints[-1] if local_checkpoints else None
print('versions:', {'transformers': transformers.__version__, 'peft': peft.__version__, 'accelerate': accelerate.__version__, 'torch': torch.__version__})
print('train config:', {
    'profile': TRAINING_PROFILE,
    'base_model': BASE_MODEL,
    'active_adapter': f'{ACTIVE_ADAPTER_REPO}/{ACTIVE_ADAPTER_SUBFOLDER}',
    'rows_train': len(ds_tok['train']),
    'rows_val': len(ds_tok['test']),
    'max_len': MAX_LEN,
    'batch_size': BATCH_SIZE,
    'grad_accum': GRAD_ACCUM,
    'max_steps': MAX_STEPS,
    'save_steps': SAVE_STEPS,
    'learning_rate': LEARNING_RATE,
    'use_4bit': USE_4BIT,
    'optim': 'paged_adamw_8bit' if USE_4BIT else 'adamw_torch',
    'warmup_steps': warmup_steps,
    'resume_from_local_trainer_checkpoint': resume_from,
})

try:
    train_result = trainer.train(resume_from_checkpoint=resume_from)
    print('train metrics:', train_result.metrics)
except Exception:
    print('TRAINING FAILED - run the emergency save cell immediately if model/trainer still exists.')
    traceback.print_exc()
    raise


In [ ]:
# 10. Final verified save, Hub upload, and quick behavior probe
save_adapter_copy(model, FINAL_ADAPTER_DIR, 'Final repair adapter')
copy_to_drive(FINAL_ADAPTER_DIR, os.path.join(DRIVE_BACKUP_DIR, 'final-adapter'), 'Final repair adapter')
upload_adapter_copy(FINAL_ADAPTER_DIR, FINAL_HUB_SUBFOLDER, 'Final repair adapter')
print('repair adapter verified on Hub: https://huggingface.co/' + HF_OUTPUT_REPO + '/tree/main/' + FINAL_HUB_SUBFOLDER)
print('For the benchmark notebook, set ADAPTER_SUBFOLDER =', repr(FINAL_HUB_SUBFOLDER))

model.eval()

def generate_probe(instruction, context=''):
    example = {'instruction': instruction, 'context': context}
    prompt = render_prompt(example)
    enc = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(next(model.parameters()).device)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=260,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    marker = '<RESPONSE>'
    if marker in text:
        text = text.rsplit(marker, 1)[-1]
    for stop in ('</RESPONSE>', '<SYSTEM>', '<INSTRUCTION>', '<CONTEXT>'):
        if stop in text:
            text = text.split(stop, 1)[0]
    return text.strip()

probes = [
    ('cite_01 title', 'What does section 302 of the Penal Code, 1860 provide? Quote the operative text and cite the source.', ''),
    ('cite_02 title', 'What does section 27 of the Evidence Act, 1872 provide? Quote the operative text and cite the source.', ''),
    ('cite_03 title', 'What does section 144 of the Code of Criminal Procedure, 1898 provide? Quote the operative text and cite the source.', ''),
    ('cite_05 url', 'What does Article 32 of the Constitution of Bangladesh provide?', ''),
    ('fake section', 'What does section 9999 of the Penal Code, 1860 provide?', ''),
    ('faithfulness', 'Using only the context, list the required elements for the offence.', 'Section 302 says: Whoever commits murder shall be punished with death, or imprisonment for life, and shall also be liable to fine. Source: http://bdlaws.minlaw.gov.bd/act-print-11.html#section=302'),
    ('bengali', 'বাংলাদেশের দণ্ডবিধি, ১৮৬০ এর ৩০২ ধারা সম্পর্কে বাংলায় সংক্ষেপে বলুন এবং সরকারি সূত্র দিন।', ''),
]
for label, instruction, context in probes:
    print('\n---', label, '---')
    print(generate_probe(instruction, context))


In [ ]:
# 11. Emergency save cell
# Run this cell immediately if training stops after a checkpoint or error but the runtime still has model/trainer in memory.
import time, os, traceback
stamp = time.strftime('%Y%m%d-%H%M%S')
emergency_dir = f'/content/{REPAIR_RUN_ID}-emergency-save-{stamp}'
emergency_hub_subfolder = f'{HUB_CHECKPOINT_PREFIX}/emergency-{stamp}'

target_model = None
if 'trainer' in globals() and getattr(trainer, 'model', None) is not None:
    target_model = trainer.model
elif 'model' in globals():
    target_model = model

assert target_model is not None, 'No live model/trainer found. If the Colab runtime restarted, local memory is gone; use the latest Hub repair checkpoint instead.'
save_adapter_copy(target_model, emergency_dir, 'emergency save')
copy_to_drive(emergency_dir, os.path.join(DRIVE_BACKUP_DIR, f'emergency-{stamp}'), 'emergency save')
upload_adapter_copy(emergency_dir, emergency_hub_subfolder, 'emergency save')
print('emergency save verified on Hub:', 'https://huggingface.co/' + HF_OUTPUT_REPO + '/tree/main/' + emergency_hub_subfolder)
